In [1]:
import pandas as pd
import numpy as np

# Load Titanic dataset from raw folder
df = pd.read_csv('../data/raw/titanic.csv')
print("Dataset Shape:", df.shape)
display(df.head())
df.info()

Dataset Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [2]:
# Target Variable Selection for Titanic
target = 'Survived'  # 1 = Survived, 0 = Did not survive
print(f"Selected Target Variable: {target}")

# Drop target and unnecessary identifier/text columns
drop_cols = [target, 'PassengerId', 'Name', 'Ticket', 'Cabin']
X = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')
y = df[target]

Selected Target Variable: Survived


In [3]:
# Feature Transformers & Preprocessor Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns

# Numerical transformer: Median imputation + Standard Scaling
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical transformer: Most frequent imputation + One-Hot Encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [4]:
# Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (712, 7)
Testing set shape: (179, 7)


In [5]:
# Train and Evaluate Logistic Regression and Random Forest
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Create a Pipeline for Logistic Regression
log_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_pipeline.fit(X_train, y_train)
y_pred_lr = log_reg_pipeline.predict(X_test)

print("--- Logistic Regression Results ---")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# 2. Create a Pipeline for Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("\n--- Random Forest Results ---")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

--- Logistic Regression Results ---
Accuracy: 0.8044692737430168
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179


--- Random Forest Results ---
Accuracy: 0.8156424581005587
              precision    recall  f1-score   support

           0       0.82      0.89      0.86       110
           1       0.80      0.70      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.81      0.82      0.81       179



In [6]:
# Ensure directories exist and save processed data splits
import os

os.makedirs('../data/processed', exist_ok=True)

train_data = X_train.copy()
train_data['Survived'] = y_train

test_data = X_test.copy()
test_data['Survived'] = y_test

train_data.to_csv('../data/processed/train_processed.csv', index=False)
test_data.to_csv('../data/processed/test_processed.csv', index=False)
print("Processed data saved successfully in data/processed/!")

Processed data saved successfully in data/processed/!


In [7]:
# Ensure directories exist and save processed data splits
import os

os.makedirs('../data/processed', exist_ok=True)

train_data = X_train.copy()
train_data['Survived'] = y_train

test_data = X_test.copy()
test_data['Survived'] = y_test

train_data.to_csv('../data/processed/train_processed.csv', index=False)
test_data.to_csv('../data/processed/test_processed.csv', index=False)
print("Processed data saved successfully in data/processed/!")

Processed data saved successfully in data/processed/!


In [8]:
# Model Comparison with Dummy Baseline & Cross-Validation
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_score, recall_score, f1_score

models = {
    "Baseline (Dummy)": DummyClassifier(strategy="most_frequent", random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

evaluation_results = []

for name, model in models.items():
    clf_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    clf_pipeline.fit(X_train, y_train)
    y_pred = clf_pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    cv_scores = cross_val_score(clf_pipeline, X_train, y_train, cv=5, scoring='accuracy')
    
    evaluation_results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "CV Mean Accuracy": round(cv_scores.mean(), 4)
    })

results_df = pd.DataFrame(evaluation_results)
print("\n--- TITANIC SURVIVAL MODEL COMPARISON ---")
display(results_df)


--- TITANIC SURVIVAL MODEL COMPARISON ---


,Model,Accuracy,Precision,Recall,F1-Score,CV Mean Accuracy
0,Baseline (Dummy),0.6145,0.0000,0.0000,0.0000,0.6166
1,Logistic Regression,0.8045,0.7931,0.6667,0.7244,0.7964
2,Random Forest,0.8156,0.8000,0.6957,0.7442,0.7908


In [9]:
import os
import joblib

# Folder create karein agar nahi hai
os.makedirs('../models', exist_ok=True)

# Agar aapne model ka variable 'model' rakha hai toh yeh save ho jayega:
joblib.dump(model, '../models/best_model.pkl')
print("Model successfully saved!")

Model successfully saved!
